# CogniVoice Component D — Sinhala retrain (MELD + collected Sinhala)

Retrains ONLY the fusion head. Reuses the already-extracted MELD features
(`features_meld.npz` in Drive), so the slow 13.7k-clip extraction is NOT
repeated — we only extract features for the 26 new Sinhala clips, then train.

**Research question:** does adding real Sinhala training data (7 new speakers)
help the FROZEN encoder generalise to UNSEEN Sinhala speakers? The held-out
eval speakers (sinhala_p1/p2/p3) are never trained on, so this is a clean
speaker-independent test. The honest before/after eval is run LOCALLY afterwards
with `scripts/evaluate_sinhala.py`.

**You need in Drive/MyDrive/cognivoice:** `features_meld.npz` (already there) and
`sinhala_collected.zip` (upload it — built locally by Claude).

Runtime → Change runtime type → **T4 GPU**.


In [ ]:
# 1. Mount Drive (has features_meld.npz + sinhala_collected.zip)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/cognivoice'
assert os.path.exists(f'{DRIVE}/features_meld.npz'), 'features_meld.npz missing in Drive/cognivoice'
assert os.path.exists(f'{DRIVE}/sinhala_collected.zip'), 'upload sinhala_collected.zip to Drive/cognivoice first'
print('Drive OK:', DRIVE)


In [ ]:
# 2. Get project code + deps (~3 min)
%cd /content
!rm -rf cognivoice-component-d
!git clone -q -b phase1-valence-primary-scoring https://github.com/Prathikesh/cognivoice-component-d.git
%cd cognivoice-component-d
!pip install -q funasr modelscope librosa soundfile praat-parselmouth scipy scikit-learn tqdm
print('setup done')


In [ ]:
# 3. Bring in the data: cached MELD features + the 26 Sinhala wavs
import os
os.makedirs('data', exist_ok=True); os.makedirs('models', exist_ok=True)
# MELD features (already extracted - do NOT re-extract)
!cp '{DRIVE}/features_meld.npz' data/features_meld.npz
# Sinhala wavs + metadata (data/ is gitignored, so they arrive via this zip)
!unzip -q -o '{DRIVE}/sinhala_collected.zip' -d .
# Regenerate the Sinhala metadata from filenames (deterministic; safety net)
!python -m src.datasets.sinhala_collected --root data/raw/real_collected_sinhala --out data/metadata_sinhala_collected.csv


In [ ]:
# 4. FEATURE EXTRACTION for the 26 Sinhala clips only (fast: ~1-2 min on T4).
# AUGMENT adds that many phone-like copies per TRAIN clip (robustness, not new
# speakers). Run AUGMENT=0 first for the clean result, then try 3 as a second
# lever and compare both against baseline. Outputs are named by augment level.
AUGMENT = 0
suffix = f'_aug{AUGMENT}' if AUGMENT else ''
SINHALA_FEATS = f'data/features_sinhala{suffix}.npz'
!python scripts/extract_features.py --metadata data/metadata_sinhala_collected.csv --out {SINHALA_FEATS} --encoder plus_large --augment {AUGMENT}
!cp {SINHALA_FEATS} '{DRIVE}/'
print('sinhala features ->', SINHALA_FEATS)


In [ ]:
# 5. TRAIN the fusion head on MELD + Sinhala (minutes). MELD supplies the
# train/val/test split; all Sinhala clips are train. Best checkpoint chosen on
# MELD val CCC. The Sinhala eval speakers are NOT in here - eval is separate.
CKPT = f'models/fusion_meld_sinhala{suffix}.pt'
!python scripts/train_fusion.py --features data/features_meld.npz {SINHALA_FEATS} --out {CKPT}


In [ ]:
# 6. Save the new checkpoint + history back to Drive
!cp {CKPT} '{DRIVE}/'
hist = CKPT.replace('.pt', '.history.json')
!cp {hist} '{DRIVE}/' 2>/dev/null || true
print('saved', CKPT, 'to Drive - download it into models/ on your Mac')


## Next (run locally on the Mac)

Download `fusion_meld_sinhala.pt` (and any `_aug3` variant) from Drive into
`models/`, then run the honest before/after eval on the held-out Sinhala speakers:

```bash
.venv/bin/python scripts/evaluate_sinhala.py \
    --model models/fusion_meld_baseline.pt models/fusion_meld_sinhala.pt \
    --metadata data/metadata_sinhala.csv
```

Baseline (before) is **90.9% acc / 87.5% stressed recall / 100% calm spec**.
Report whatever the new number is - up, down, or flat. A flat/down result is a
legitimate finding (it would corroborate the prior 'small Sinhala data doesn't
help the frozen encoder' result); an up result means the collected data helped.
